# AAPL fixed trend filter

Would a simple 200-day trend filter have improved AAPL's risk and return versus buy-and-hold over 2016–2025?

**Data and assumptions**

- AAPL vendor daily OHLC and corporate-action data; the feed does not verify exact regular-session or opening-auction prints.
- Hold AAPL when its completed daily close is above its 200-session SMA, otherwise cash. Act at the next vendor daily open; keep the rule fixed.
- Start with USD 1 million, no leverage or shorting, 10 bps each way, no additional slippage and zero cash interest. The comparator is gross dividend-inclusive buy-and-hold.

**Requested answer:** performance (table), strategy_summary (json), trader_conclusion (plain).

This notebook contains its complete request and setup. Share this file; no local case JSON or helper module is required. It contains no pre-filled market results.

## Connect

Select a Python kernel with `epoch-sdk[notebook]` installed. On the prepared
server, choose **Epoch SDK (local prototype)**.

The notebook needs only your Epoch login and the two service addresses. The
collapsed helper below obtains a current identity token from the environment's
configured Epoch login command, or uses `EPOCH_TOKEN`. It prints no credentials
and reads no service configuration files. On another machine, supply your own
`EPOCH_TOKEN`, `EPOCH_BACKEND_URL` and `EPOCH_GRAPH_URL`.

The next cell prepares the client and checks both services. It starts no research.

In [ ]:
"""Obtain only an Epoch bearer token from the local operator's login provider."""

from __future__ import annotations

import json
import os
import subprocess


def identity_token() -> str:
    """Read a static token or ask the configured login helper for a current one.

    The prepared notebook launcher configures a trusted argv list in the
    environment. It uses the owner's normal Epoch login/refresh flow. No shell
    is used, and neither provider output nor credentials appear in exceptions.
    Standalone notebook environments can continue to supply EPOCH_TOKEN alone.
    """
    command = os.environ.get("EPOCH_NOTEBOOK_TOKEN_COMMAND")
    if command:
        try:
            argv = json.loads(command)
        except (TypeError, ValueError):
            raise RuntimeError(
                "The notebook's Epoch login command is invalid."
            ) from None
        if (
            not isinstance(argv, list)
            or not argv
            or not all(isinstance(value, str) and value for value in argv)
        ):
            raise RuntimeError(
                "The notebook's Epoch login command must be a JSON argv list."
            )
        try:
            result = subprocess.run(
                argv, capture_output=True, text=True, timeout=30, check=False
            )
        except (OSError, subprocess.TimeoutExpired):
            raise RuntimeError(
                "Could not contact the local Epoch login helper. Restore the normal "
                "Epoch login and rerun this setup cell."
            ) from None
        token = result.stdout.strip()
        if result.returncode or not token or any(value.isspace() for value in token):
            raise RuntimeError(
                "The Epoch login needs attention. Sign in through the normal login "
                "flow, then rerun this setup cell."
            )
        return token
    token = os.environ.get("EPOCH_TOKEN")
    if not token:
        raise RuntimeError(
            "Set EPOCH_TOKEN to your Epoch identity token, or use the prepared local "
            "notebook server, which supplies Epoch login to its kernels."
        )
    return token

In [ ]:
import importlib.metadata
import httpx
from IPython.display import JSON, display
from epoch_sdk import RunRequest
from epoch_sdk.research import EpochClient, ResearchCompleted

# Connect to Epoch using your login, not vendor or model credentials.
_epoch_token = identity_token()
epoch = EpochClient.connect(
    graph_url=os.environ["EPOCH_GRAPH_URL"],
    backend_url=os.environ["EPOCH_BACKEND_URL"],
    token=_epoch_token,
)

print("Epoch SDK:", importlib.metadata.version("epoch-sdk"))
print("Native SDK:", importlib.metadata.version("stratifyx"))
async with httpx.AsyncClient(
    headers={"Authorization": "Bearer " + _epoch_token}, timeout=20,
) as health:
    for name, origin, route in [
        ("Backend", os.environ["EPOCH_BACKEND_URL"], "/api/v1/health"),
        ("EpochAI", os.environ["EPOCH_GRAPH_URL"], "/health/custom"),
    ]:
        response = await health.get(origin.rstrip("/") + route)
        response.raise_for_status()
        print(name + ": ready")
del _epoch_token

## The request

The task and context are the desk request shown above. The collapsed output
schema declares the requested fields, formats and units; expand it to inspect
or change them. It contains no result values.

Keep the attempt unchanged to reconnect, including after a kernel restart.
Choose a new attempt when intentionally changing the request or refreshing data.

In [ ]:
OUTPUTS = [{'name': 'performance',
  'type': 'table',
  'description': 'Executed strategy versus the gross analytical buy-and-hold benchmark.',
  'schema': [{'name': 'variant',
              'type': 'string',
              'description': 'trend_filter or buy_and_hold; in that order.'},
             {'name': 'total_return',
              'type': 'float64',
              'description': 'Cumulative account return for the strategy; gross total return for '
                             'the analytical benchmark.',
              'unit': 'fraction'},
             {'name': 'annualized_return',
              'type': 'float64',
              'description': 'Compound annual growth rate over the common evaluation window.',
              'unit': 'fraction'},
             {'name': 'annualized_volatility',
              'type': 'float64',
              'description': 'Sample standard deviation of daily account returns times sqrt(252).',
              'unit': 'fraction'},
             {'name': 'maximum_drawdown',
              'type': 'float64',
              'description': 'Minimum wealth / running peak - 1; non-positive.',
              'unit': 'fraction'}]},
 {'name': 'strategy_summary',
  'type': 'json',
  'description': 'The same performance values plus the fixed rule and cost assumptions.',
  'schema': {'type': 'object',
             'description': 'Fixed-rule strategy summary.',
             'properties': {'asset': {'type': 'string',
                                      'description': 'US ticker.',
                                      'const': 'AAPL'},
                            'sma_sessions': {'type': 'integer',
                                             'description': 'Fixed moving-average length.',
                                             'const': 200},
                            'trade_timing': {'type': 'string',
                                             'description': 'Orders use the next available vendor '
                                                            'daily open; no verified '
                                                            'opening-auction claim.',
                                             'const': 'next_session_open'},
                            'transaction_cost_bps': {'type': 'number',
                                                     'description': 'One-way all-in modeled native '
                                                                    'commission rate.',
                                                     'const': 10},
                            'cash_interest_rate': {'type': 'number',
                                                   'description': 'Annual cash interest '
                                                                  'assumption.',
                                                   'const': 0},
                            'benchmark_basis': {'type': 'string',
                                                'description': 'Gross analytical benchmark '
                                                               'convention.',
                                                'const': 'gross_buy_and_hold_total_return'},
                            'strategy': {'type': 'object',
                                         'description': 'Actual executed-account metrics, matching '
                                                        'the trend_filter table row.',
                                         'properties': {'total_return': {'type': 'number',
                                                                         'description': 'Cumulative '
                                                                                        'account '
                                                                                        'return '
                                                                                        'for the '
                                                                                        'strategy; '
                                                                                        'gross '
                                                                                        'total '
                                                                                        'return '
                                                                                        'for the '
                                                                                        'analytical '
                                                                                        'benchmark. '
                                                                                        'Stored as '
                                                                                        'a '
                                                                                        'fraction.',
                                                                         'minimum': -1},
                                                        'annualized_return': {'type': 'number',
                                                                              'description': 'Compound '
                                                                                             'annual '
                                                                                             'growth '
                                                                                             'rate '
                                                                                             'over '
                                                                                             'the '
                                                                                             'common '
                                                                                             'evaluation '
                                                                                             'window. '
                                                                                             'Stored '
                                                                                             'as a '
                                                                                             'fraction.',
                                                                              'minimum': -1},
                                                        'annualized_volatility': {'type': 'number',
                                                                                  'description': 'Sample '
                                                                                                 'standard '
                                                                                                 'deviation '
                                                                                                 'of '
                                                                                                 'daily '
                                                                                                 'account '
                                                                                                 'returns '
                                                                                                 'times '
                                                                                                 'sqrt(252). '
                                                                                                 'Stored '
                                                                                                 'as '
                                                                                                 'a '
                                                                                                 'fraction.',
                                                                                  'minimum': 0},
                                                        'maximum_drawdown': {'type': 'number',
                                                                             'description': 'Minimum '
                                                                                            'wealth '
                                                                                            '/ '
                                                                                            'running '
                                                                                            'peak '
                                                                                            '- 1; '
                                                                                            'non-positive. '
                                                                                            'Stored '
                                                                                            'as a '
                                                                                            'fraction.',
                                                                             'minimum': -1,
                                                                             'maximum': 0}},
                                         'required': ['total_return',
                                                      'annualized_return',
                                                      'annualized_volatility',
                                                      'maximum_drawdown'],
                                         'additionalProperties': False},
                            'benchmark': {'type': 'object',
                                          'description': 'Analytical benchmark metrics, matching '
                                                         'the buy_and_hold table row.',
                                          'properties': {'total_return': {'type': 'number',
                                                                          'description': 'Cumulative '
                                                                                         'account '
                                                                                         'return '
                                                                                         'for the '
                                                                                         'strategy; '
                                                                                         'gross '
                                                                                         'total '
                                                                                         'return '
                                                                                         'for the '
                                                                                         'analytical '
                                                                                         'benchmark. '
                                                                                         'Stored '
                                                                                         'as a '
                                                                                         'fraction.',
                                                                          'minimum': -1},
                                                         'annualized_return': {'type': 'number',
                                                                               'description': 'Compound '
                                                                                              'annual '
                                                                                              'growth '
                                                                                              'rate '
                                                                                              'over '
                                                                                              'the '
                                                                                              'common '
                                                                                              'evaluation '
                                                                                              'window. '
                                                                                              'Stored '
                                                                                              'as '
                                                                                              'a '
                                                                                              'fraction.',
                                                                               'minimum': -1},
                                                         'annualized_volatility': {'type': 'number',
                                                                                   'description': 'Sample '
                                                                                                  'standard '
                                                                                                  'deviation '
                                                                                                  'of '
                                                                                                  'daily '
                                                                                                  'account '
                                                                                                  'returns '
                                                                                                  'times '
                                                                                                  'sqrt(252). '
                                                                                                  'Stored '
                                                                                                  'as '
                                                                                                  'a '
                                                                                                  'fraction.',
                                                                                   'minimum': 0},
                                                         'maximum_drawdown': {'type': 'number',
                                                                              'description': 'Minimum '
                                                                                             'wealth '
                                                                                             '/ '
                                                                                             'running '
                                                                                             'peak '
                                                                                             '- 1; '
                                                                                             'non-positive. '
                                                                                             'Stored '
                                                                                             'as a '
                                                                                             'fraction.',
                                                                              'minimum': -1,
                                                                              'maximum': 0}},
                                          'required': ['total_return',
                                                       'annualized_return',
                                                       'annualized_volatility',
                                                       'maximum_drawdown'],
                                          'additionalProperties': False}},
             'required': ['asset',
                          'sma_sessions',
                          'trade_timing',
                          'transaction_cost_bps',
                          'cash_interest_rate',
                          'benchmark_basis',
                          'strategy',
                          'benchmark'],
             'additionalProperties': False}},
 {'name': 'trader_conclusion',
  'type': 'text',
  'format': 'plain',
  'instructions': 'Write at most 120 words. State whether the fixed trend filter reduced drawdown '
                  'and what return trade-off occurred, quoting computed figures. Mention next '
                  'vendor-open execution, 10 bps each way, zero cash interest and the gross '
                  "benchmark. Disclose that the daily feed's exact session/auction basis is not "
                  'independently established. Do not imply that a successful run validates a '
                  'trading edge. No Markdown table.'}]

In [ ]:
TASK = "Would a simple 200-day trend filter have improved AAPL's risk and return versus buy-and-hold over 2016–2025?"
CONTEXT = ['AAPL vendor daily OHLC and corporate-action data; the feed does not verify exact regular-session '
 'or opening-auction prints.',
 'Hold AAPL when its completed daily close is above its 200-session SMA, otherwise cash. Act at '
 'the next vendor daily open; keep the rule fixed.',
 'Start with USD 1 million, no leverage or shorting, 10 bps each way, no additional slippage and '
 'zero cash interest. The comparator is gross dividend-inclusive buy-and-hold.']

PREFIX = os.environ.get("EPOCH_NOTEBOOK_PREFIX", "notebook-demo")
ATTEMPT = "desk-v2-01"
RESEARCH_ID = f"{PREFIX}-aapl_trend_strategy-{ATTEMPT}-research"
PARAMETERS = {'period': {'start': '2016-01-01', 'end': '2025-12-31'},
 'execution': {'cash': 1000000,
               'currency': 'USD',
               'commission': {'model': {'type': 'Bps', 'args': {'bps': 10.0}}},
               'slippage': {'model': {'type': 'None', 'args': {}}}}}
request = RunRequest.model_validate({
    "task": TASK, "context": CONTEXT, "parameters": PARAMETERS, "outputs": OUTPUTS,
})
print("Research ID:", RESEARCH_ID)
display(JSON(request.to_wire(), expanded=False))

## Run the session

This is the cell that starts research or reconnects to the same operation. It
shows live activity and the published answer. If Epoch asks a question, choose
an option or write a reply and press **Send reply** inside the cell.

Nothing is selected automatically. If you leave a question waiting, rerun this
cell with the same research ID before continuing to the result cells.

In [ ]:
# The SDK owns streaming, questions and resume.
result = await epoch.run(
    request,
    operation_id=RESEARCH_ID,
)

## Use the result in Python

The session has already displayed the answer. These cells read its typed parts
for further analysis without printing the same narrative again. Table reads
retrieve every page; JSON and text are already in the saved answer.

In [ ]:
assert isinstance(result, ResearchCompleted), "Resume the session above first."
answer = result.response
json_answers = {part.name: part.value for part in answer.content.outputs if part.type == "json"}
text_answers = {part.name: part.value for part in answer.content.outputs if part.type == "text"}
# Read every page; a page size does not limit the total table size.
tables = {}
for part in result.response.content.outputs:
    if part.type != "table":
        continue
    rows = []
    async for page in epoch.iter_response_table(
        result.response.response_id,
        part.name,
        version=result.response.version,
        page_size=1000,
    ):
        names = [column.name for column in page.columns]
        rows.extend(dict(zip(names, row, strict=True)) for row in page.rows)
    assert len(rows) == part.source.row_count
    tables[part.name] = rows
print("Saved answer:", answer.response_id, "revision", answer.version)
print("Complete table rows:", {name: len(rows) for name, rows in tables.items()})

## Optional: execute the saved study again

This separate action creates a new Backend execution of the saved study using
the same period and costs. It does not rediscover data, rebuild the study,
refresh a saved roster or publish a new interpretation. It currently returns
execution status; response-plan replay remains separate work.

The execution ID below is independent of the research ID. Repeating the call
with the same ID attaches to the same execution.

In [ ]:
EXECUTION_ID = f"{PREFIX}-aapl_trend_strategy-{ATTEMPT}-execute"
assert result.reusable_study is not None
execution = await epoch.execute_saved(
    result.reusable_study,
    period=request.parameters.period,
    execution=request.parameters.execution,
    operation_id=EXECUTION_ID,
)
print("Execution operation:", EXECUTION_ID)
# Read this same ID later to get its current native status.
execution_status = await epoch.execution_operation(EXECUTION_ID)

## Close this client

Saved work remains on the services. Keep the readable research ID to reconnect.

In [ ]:
await epoch.aclose()